## Add forcing conditions to model from Python

In this example, the previously made **SFINCS** compound flood model will be updated with new boundary conditions..

The model is situated in **Northern Italy**, where a small selection of topography and bathymetry data has already been made available for you to try the examples.

In [ ]:
from datetime import datetime

from hydromt_sfincs import SfincsModel
from hydromt._utils import log

# Initialize logging, the lower the log level number, the more verbose (more info) the output
# NOTSET=0-9, DEBUG=10, INFO=20, WARNING=30, ERROR=40, CRITICAL=50

log.initialize_logging()
log.set_log_level(log_level=20)

**Steps followed in this notebook to update your SFINCS model:**
<ol> 
<li> Initialize SfincsModel class, set data library, mode and root folder </li>
<li> Add water level time-series as forcing</li>
<li> Add an upstream discharge time-series as forcing</li>
<li> Add spatially varying rainfall data</li>
<li> Save all files</li>
</ol> 

Let's get started!

### 1. Initialize SfincsModel class, set data library, mode and root folder:

Before we can use all the tools provided by HydroMT-SFINCS, we have to initialize the SfincsModel instance. This creates a shortcut to all the model components and methods to read, write and create these components.

In contrast to the previous notebook, we now initialize the model in "append" mode: "r+", to make sure we can upgrade some of its components.

In [ ]:
# Initialize SfincsModel Python class with the artifact data catalog which contains publically available data for North Italy
sf = SfincsModel(
    data_libs=["artifact_data"],  # specify which data libraries to use
    root="tmp_sfincs_compound",  # specify the root directory for the model
    mode="r+",  # specify the mode for opening the model (r=read only, r+=append, w=write, w+=overwrite
    write_gis=True,  # specify whether to write GIS data
)


### 2. Add water level time-series as forcing:
There are several ways to add **water level boundary conditions** to the model:

1. **🌐 Using a geodataset (presented here)**  
   - The simplest option.  
   - Contains time series of water levels at specific points.  
   - Often generated by other models, such as the **GTSM model**.

2. **📊 Using shapefile/GeoJSON + CSV**  
   - Locations are specified in a shapefile or GeoJSON.  
   - Corresponding time series are read from a CSV file.

3. **⚙️ Manually generating artificial time series**  
   - Points are manually placed along the water level boundary.  
   - Artificial time series can be created for each point.

In [ ]:
# Change period of model simulation time, specified in yyyymmdd HHMMSS, to match with the available water level data
sf.config.update(
    {
        "tref": datetime(2010, 2, 5),
        "tstart": datetime(2010, 2, 5),
        "tstop": datetime(2010, 2, 7),
    }
)

# From geodataset
sf.water_level.create(geodataset="gtsmv3_eu_era5")

# Along water level boundary with synthetic sinusoidal data
# sf.water_level.create_boundary_points_from_mask()
# sf.water_level.create_timeseries(shape="constant", offset=2.0)

To inspect the added boundary conditions, we can again use the plotting functionality of HydroMT-SFINCS. When the boundary conditions are related to boundary points (water levels and waves), these locations can be visualzied with `SfincsModel.plot_basemap(plot_geoms=True)`. Nonetheless, the actual timeseries of the boundary conditions can be visualized with `SfincsModel.plot_forcing()`.

In [ ]:
# plot basemap, in this case boundary points should be added (and plotted because of plot_geoms=True)
sf.plot_basemap(
    variable="dep", plot_geoms=True, plot_bounds=True, bmap="sat", zoomlevel=12
)

And the boundary conditions timeseries:

In [ ]:
sf.plot_forcing()

<div style="border-left: 4px solid #4CAF50; padding: 0.5em; background-color: #f0fff0;">
<b>⚠️ Note:</b> The discharge timeseries show 0s since we did not specify them yet.
</div>.

### 3. Add an upstream discharge time-series as forcing:
As could be seen in the plotted timeseries above, the discharge forcing for the two inflow points are not yet specified. Similar to the water levels, there is many ways to specify the [discharge points](https://sfincs.readthedocs.io/en/latest/input_forcing.html#discharge-points) and the [discahrge timeseries](https://sfincs.readthedocs.io/en/latest/input_forcing.html#discharge-time-series), but only one will be discussed here.

In [ ]:
sf.discharge_points.create_timeseries(
    index=[0, 1],
    shape="gaussian",
    offset=0,
    peak=5,
    tpeak=86400,
    duration=2 * 86400,
    timestep=3600,
)
# and plot
sf.plot_forcing()

# optionally, points can be added manually
# sf.discharge_points.add_point(
#     x=322030.6 - 50, y=5047006.9 + 50, value=1000.0, name="test_point"
# )

### 4. Add spatially varying rainfall data:

In [ ]:
# # hourly rainfall rates of ECMWF' ERA5 data for the specific area and period have been made available for this period in the artefact data
sf.precipitation.create(precip="era5_hourly", aggregate=False, buffer=30e3)

# NOTE: when specifying an output name, the image is also saved to file
sf.plot_forcing(fn_out="forcing.png")

<div style="border-left: 4px solid #4CAF50; padding: 10px; margin: 5px 0; background-color: #f0fff0;">
<b>💡 Tip:</b> In case you want to add other types of forcing, read more in the <a href="https://sfincs.readthedocs.io/en/latest/input_forcing.html" target="_blank">SFINCS manual</a>.
</div>


### 5. Write all files

In [ ]:
sf.write()  # write all

Now your model has boundary conditions, you can progress to the notebook: [4. Run SFINCS model](4_run_model.ipynb) or add some geometries and structures to your model first in [3. Add Geometries](3_add_geometries.ipynb)